In [1]:
# ============ TIR Binary Classification Model (v2) ============
# Clean version with balanced undersampling
# Task: Predict TRUE (has TIR) vs FALSE (no TIR)

import os
import gc
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix, accuracy_score, balanced_accuracy_score,
                             f1_score, classification_report)

# ============ Configuration ============
FASTA_PATH = "../../data/vgp/all_vgp_tes.fa"
LABEL_PATH = "../../data/vgp/features"
FIXED_LENGTH = 40000
RC_FUSION_MODE = "early"

# Training parameters
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-4  # Lower LR for stability on GPU/MPS
PATIENCE = 10
MAX_SEQ_PER_CLASS = 15000  # For balanced subset creation

def resolve_device(requested=None):
    if requested is not None:
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

Using device: mps


## Data Loading & Balanced Sampling

In [2]:
def read_fasta(path):
    """Read FASTA file and return headers and sequences."""
    headers, sequences = [], []
    h, buf = None, []
    with open(path, 'r') as f:
        for line in f:
            if not line:
                continue
            if line[0] == '>':
                if h is not None:
                    sequences.append(''.join(buf).upper())
                    buf = []
                h = line[1:].strip()
                headers.append(h)
            else:
                buf.append(line.strip())
        if h is not None:
            sequences.append(''.join(buf).upper())
    return headers, sequences


def load_tir_labels(label_path):
    """Load TIR presence/absence labels."""
    label_path = Path(label_path)
    label_dict = {}
    n_true, n_false = 0, 0
    
    with label_path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            header = parts[0].lstrip('>')
            tag = parts[1].upper()
            
            if tag == "TRUE":
                label_dict[header] = 1
                n_true += 1
            elif tag == "FALSE":
                label_dict[header] = 0
                n_false += 1
    
    total = n_true + n_false
    print(f"Loaded {total} TIR labels:")
    print(f"  TRUE (has TIR):  {n_true} ({100*n_true/total:.1f}%)")
    print(f"  FALSE (no TIR):  {n_false} ({100*n_false/total:.1f}%)")
    return label_dict


def create_balanced_subset(headers, sequences, labels, max_num=None, random_state=42):
    """Undersample both classes to create a balanced subset."""
    np.random.seed(random_state)
    labels = np.asarray(labels)
    
    idx_false = np.where(labels == 0)[0]
    idx_true = np.where(labels == 1)[0]
    n_false_orig = len(idx_false)
    n_true_orig = len(idx_true)
    
    print(f"\nOriginal: {n_false_orig} FALSE / {n_true_orig} TRUE")
    
    # Determine target size per class
    target_per_class = min(n_false_orig, n_true_orig)
    if max_num is not None:
        target_per_class = min(target_per_class, max_num)
    
    print(f"Target per class: {target_per_class}")
    
    # Sample from both classes
    idx_false_sampled = np.random.choice(idx_false, size=target_per_class, replace=False)
    idx_true_sampled = np.random.choice(idx_true, size=target_per_class, replace=False)
    
    idx_balanced = np.concatenate([idx_false_sampled, idx_true_sampled])
    np.random.shuffle(idx_balanced)
    
    balanced_h = [headers[i] for i in idx_balanced]
    balanced_s = [sequences[i] for i in idx_balanced]
    balanced_labels = labels[idx_balanced]
    
    print(f"Balanced: {target_per_class} FALSE / {target_per_class} TRUE = {2*target_per_class} total")
    return balanced_h, balanced_s, balanced_labels

## Dataset & Encoding

In [3]:
# Encoding: ACGT -> 0-3, N -> 4
ENCODE = np.full(256, 4, dtype=np.int64)
for ch, idx in zip(b"ACGTNacgtn", [0, 1, 2, 3, 4, 0, 1, 2, 3, 4]):
    ENCODE[ch] = idx

REV_COMP = torch.tensor([3, 2, 1, 0, 4], dtype=torch.long)


class TIRDataset(Dataset):
    """Dataset with random placement augmentation."""
    def __init__(self, headers, sequences, labels, fixed_length=FIXED_LENGTH):
        self.headers = list(headers)
        self.sequences = list(sequences)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.fixed_length = fixed_length
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        seq_len = len(seq)
        seq_bytes = seq.encode("ascii", "ignore")
        seq_idx = ENCODE[np.frombuffer(seq_bytes, dtype=np.uint8)]
        
        max_start = max(0, self.fixed_length - seq_len)
        start_pos = np.random.randint(0, max_start + 1) if max_start > 0 else 0
        
        return self.headers[idx], seq_idx, int(self.labels[idx]), start_pos, seq_len


def collate_tir(batch, fixed_length=FIXED_LENGTH):
    headers, seq_idxs, labels, starts, lengths = zip(*batch)
    B = len(batch)
    X = torch.zeros((B, 5, fixed_length), dtype=torch.float32)
    mask = torch.zeros((B, fixed_length), dtype=torch.bool)
    
    for i, (seq_idx, start, seq_len) in enumerate(zip(seq_idxs, starts, lengths)):
        actual_len = min(seq_len, fixed_length - start)
        if actual_len > 0:
            idx = torch.from_numpy(seq_idx[:actual_len].astype(np.int64))
            pos = torch.arange(actual_len, dtype=torch.long) + start
            X[i, idx, pos] = 1.0
            mask[i, start:start + actual_len] = (idx != 4)
    
    Y = torch.tensor(labels, dtype=torch.long)
    return list(headers), X, mask, Y

## Model Architecture

In [4]:
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel_size=9, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size // 2) * dilation
        self.conv = nn.Conv1d(c_in, c_out, kernel_size, padding=pad, dilation=dilation)
        self.bn = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Identity() if c_in == c_out else nn.Conv1d(c_in, c_out, 1)

    def forward(self, x):
        y = F.gelu(self.bn(self.conv(x)))
        return self.drop(y) + self.proj(x)


class MaskedMaxPool1d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride

    def forward(self, x, mask):
        if mask is not None:
            x = x * mask.unsqueeze(1).float() + (~mask.unsqueeze(1)) * (-1e9)
        x_p = F.max_pool1d(x, self.kernel_size, self.stride)
        if mask is None:
            return x_p, None
        m_p = F.max_pool1d(mask.float().unsqueeze(1), self.kernel_size, self.stride).squeeze(1) > 0
        return x_p, m_p


def masked_avg_pool(z, mask):
    if mask is None:
        return z.mean(-1)
    m = mask.unsqueeze(1).float()
    return (z * m).sum(-1) / m.sum(-1).clamp_min(1.0)


class MaskedAttentionPooling(nn.Module):
    """Attention pooling over sequence positions with mask support."""
    def __init__(self, hidden_size):
        super().__init__()
        self.query = nn.Linear(hidden_size, hidden_size)
        self.score = nn.Linear(hidden_size, 1)

    def forward(self, x, mask=None):  # x: (B, C, L)
        # (B, L, C)
        x_t = x.transpose(1, 2)
        # (B, L)
        attn_scores = self.score(torch.tanh(self.query(x_t))).squeeze(-1)
        if mask is not None:
            # Masked softmax
            attn_scores = attn_scores.masked_fill(~mask, -1e9)
        attn = torch.softmax(attn_scores, dim=-1)
        # Weighted sum: (B, 1, L) @ (B, L, C) -> (B, C)
        pooled = torch.bmm(attn.unsqueeze(1), x_t).squeeze(1)
        return pooled


class RCFirstConv1d(nn.Module):
    """RC-invariant first convolution."""
    def __init__(self, out_channels, kernel_size=15, dropout=0.1):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv1d(5, out_channels, kernel_size, padding=pad)
        self.bn = nn.BatchNorm1d(out_channels)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y1 = self.conv(x)
        x_rc = x.flip(-1).index_select(1, REV_COMP.to(x.device))
        y2 = self.conv(x_rc).flip(-1)
        y = torch.max(y1, y2)
        return self.drop(F.gelu(self.bn(y)))


class TIRCNN(nn.Module):
    """RC-invariant CNN for TIR binary classification with attention pooling."""
    def __init__(self, width=128, motif_kernels=(7, 15, 21), 
                 context_dilations=(1, 2, 4, 8), dropout=0.15):
        super().__init__()

        # Multi-scale RC-invariant motif detection
        self.motif_convs = nn.ModuleList([
            RCFirstConv1d(width, kernel_size=k, dropout=dropout)
            for k in motif_kernels
        ])

        # Mix layer
        in_ch = width * len(motif_kernels)
        self.mix = nn.Sequential(
            nn.Conv1d(in_ch, width, 1),
            nn.BatchNorm1d(width),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Context blocks
        self.context_blocks = nn.ModuleList([
            ConvBlock(width, width, kernel_size=9, dilation=d, dropout=dropout)
            for d in context_dilations
        ])
        self.pool = MaskedMaxPool1d()

        # Attention pooling over positions
        self.attention_pool = MaskedAttentionPooling(width)

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(width, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2),
        )
    
    def forward(self, x, mask):
        # Motif feature maps
        feats = [conv(x) for conv in self.motif_convs]
        z = self.mix(torch.cat(feats, dim=1))

        # Contextualization + pooling
        m = mask
        for block in self.context_blocks:
            z = block(z)
            z, m = self.pool(z, m)

        # Attention pooling across sequence
        z_pooled = self.attention_pool(z, m)
        return self.classifier(z_pooled)

## Training Function

In [5]:
import heapq

class TopKCheckpointManager:
    """Manages top-K best checkpoints, saving them in real-time."""
    
    def __init__(self, save_dir: str, prefix: str, k: int = 5):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.prefix = prefix
        self.k = k
        # Min-heap: (neg_score, epoch) - we use neg so min-heap gives us worst
        self.heap = []
        self.checkpoints = {}  # epoch -> checkpoint path
    
    def maybe_save(self, score: float, epoch: int, model, history: dict, 
                   optimizer=None, scheduler=None):
        """Check if this epoch should be saved and save it if so."""
        neg_score = -score
        
        if len(self.heap) < self.k:
            # Not full yet, always save
            self._save_checkpoint(score, epoch, model, history, optimizer, scheduler)
            heapq.heappush(self.heap, (neg_score, epoch))
            print(f"  💾 Saved checkpoint (top {len(self.heap)}/{self.k})")
            return True
        elif neg_score < self.heap[0][0]:
            # Better than worst in top-k
            _, worst_epoch = heapq.heappop(self.heap)
            self._remove_checkpoint(worst_epoch)
            
            self._save_checkpoint(score, epoch, model, history, optimizer, scheduler)
            heapq.heappush(self.heap, (neg_score, epoch))
            print(f"  💾 Saved checkpoint (replaced epoch {worst_epoch})")
            return True
        return False
    
    def _save_checkpoint(self, score, epoch, model, history, optimizer=None, scheduler=None):
        """Save a checkpoint to disk."""
        state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        ckpt = {
            "model_state_dict": state_dict,
            "history": dict(history),
            "epoch": epoch,
            "score": score,
        }
        if optimizer is not None:
            ckpt["optimizer_state_dict"] = optimizer.state_dict()
        if scheduler is not None:
            ckpt["scheduler_state_dict"] = scheduler.state_dict()
        
        path = self.save_dir / f"{self.prefix}_epoch{epoch}.pt"
        torch.save(ckpt, path)
        self.checkpoints[epoch] = path
    
    def _remove_checkpoint(self, epoch: int):
        """Remove a checkpoint file."""
        if epoch in self.checkpoints:
            path = self.checkpoints[epoch]
            if path.exists():
                path.unlink()
            del self.checkpoints[epoch]
    
    def get_best(self):
        """Get the best checkpoint (highest score)."""
        if not self.heap:
            return None, None
        best_neg_score, best_epoch = min(self.heap, key=lambda x: x[0])
        if best_epoch in self.checkpoints:
            ckpt = torch.load(self.checkpoints[best_epoch], weights_only=False)
            return ckpt, best_epoch
        return None, best_epoch
    
    def get_all_saved_epochs(self):
        """Return list of (score, epoch) sorted by score descending."""
        result = [(-neg_score, epoch) for neg_score, epoch in self.heap]
        return sorted(result, reverse=True)


def load_checkpoint_for_resume(checkpoint_path, model, optimizer=None, scheduler=None, device=None):
    """Load a checkpoint and return the starting epoch."""
    ckpt = torch.load(checkpoint_path, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    if device:
        model.to(device)
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    print(f"Resumed from epoch {ckpt['epoch']} (score: {ckpt['score']:.4f})")
    return ckpt["epoch"], ckpt.get("history", {"train_loss": [], "val_f1": [], "val_auroc": []})


def train_tir_model(
    fasta_path, label_path,
    batch_size=32, epochs=30, lr=1e-3, patience=10,
    test_size=0.2, random_state=42, save_dir=".", device=None,
    resume_from=None  # Path to checkpoint to resume from
):
    """Train TIR classifier with balanced undersampling and epoch checkpointing."""
    device = resolve_device(device)
    print(f"Device: {device}")
    
    # Load and balance data
    print("\n=== Loading data ===")
    headers, sequences = read_fasta(fasta_path)
    label_dict = load_tir_labels(label_path)
    
    all_h, all_s, all_labels = [], [], []
    for h, s in zip(headers, sequences):
        if h in label_dict:
            all_h.append(h)
            all_s.append(s)
            all_labels.append(label_dict[h])
    
    del headers, sequences
    gc.collect()
    
    print(f"Matched {len(all_h)} sequences")
    all_labels = np.array(all_labels, dtype=np.int64)
    
    # Balance via undersampling
    all_h, all_s, all_labels = create_balanced_subset(
        all_h, all_s, all_labels, 
        max_num=MAX_SEQ_PER_CLASS, 
        random_state=random_state
    )
    
    # Train/test split
    idx_train, idx_test = train_test_split(
        np.arange(len(all_h)), test_size=test_size, stratify=all_labels, random_state=random_state
    )
    
    train_h = [all_h[i] for i in idx_train]
    train_s = [all_s[i] for i in idx_train]
    train_labels = all_labels[idx_train]
    
    test_h = [all_h[i] for i in idx_test]
    test_s = [all_s[i] for i in idx_test]
    test_labels = all_labels[idx_test]
    
    print(f"\nTrain: {len(train_h)}, Test: {len(test_h)}")
    
    # Model
    model = TIRCNN().to(device)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Warmup + cosine decay scheduler
    warmup_epochs = 3
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return 0.5 * (1 + math.cos(math.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    
    # DataLoaders
    ds_train = TIRDataset(train_h, train_s, train_labels)
    ds_test = TIRDataset(test_h, test_s, test_labels)
    
    del train_h, train_s, test_h, test_s, all_h, all_s, all_labels
    gc.collect()
    
    loader_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_tir)
    loader_test = DataLoader(ds_test, batch_size=batch_size, shuffle=False, collate_fn=collate_tir)
    
    print(f"Batches/epoch: {len(loader_train)}")
    
    # Checkpoint manager (keeps top 5 best checkpoints)
    ckpt_manager = TopKCheckpointManager(save_dir, prefix="tir_v2", k=5)
    
    # Resume from checkpoint if specified
    start_epoch = 1
    history = {"train_loss": [], "val_f1": [], "val_auroc": []}
    if resume_from is not None and Path(resume_from).exists():
        start_epoch, history = load_checkpoint_for_resume(
            resume_from, model, opt, scheduler, device
        )
        start_epoch += 1  # Start from next epoch
    
    best_f1 = max(history["val_f1"]) if history["val_f1"] else -1
    bad = 0
    
    print("\n=== Training ===")
    for ep in range(start_epoch, epochs + 1):
        model.train()
        running_loss, n_samples = 0.0, 0
        
        for _, X, mask, Y in tqdm(loader_train, desc=f"Ep {ep}/{epochs}", leave=False):
            X, mask, Y = X.to(device), mask.to(device), Y.to(device)
            
            loss = loss_fn(model(X, mask), Y)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
            running_loss += loss.item() * X.size(0)
            n_samples += X.size(0)
        
        scheduler.step()
        train_loss = running_loss / n_samples
        
        # Evaluate
        model.eval()
        all_pred, all_true, all_probs = [], [], []
        
        with torch.no_grad():
            for _, X, mask, Y in loader_test:
                X, mask = X.to(device), mask.to(device)
                logits = model(X, mask)
                all_probs.extend(F.softmax(logits, dim=1)[:, 1].cpu().numpy())
                all_pred.extend(logits.argmax(dim=1).cpu().numpy())
                all_true.extend(Y.numpy())
        
        f1 = f1_score(all_true, all_pred, average="binary")
        auroc = roc_auc_score(all_true, all_probs)
        
        # Check for mode collapse
        n_pred_0 = (np.array(all_pred) == 0).sum()
        n_pred_1 = (np.array(all_pred) == 1).sum()
        
        history["train_loss"].append(train_loss)
        history["val_f1"].append(f1)
        history["val_auroc"].append(auroc)
        
        print(f"Ep {ep:2d}: loss {train_loss:.4f} | F1 {f1:.4f} | AUROC {auroc:.4f} | pred: {n_pred_0}F/{n_pred_1}T")
        
        # Save checkpoint if in top-5
        ckpt_manager.maybe_save(
            score=f1, epoch=ep, model=model, history=history,
            optimizer=opt, scheduler=scheduler
        )
        
        # Early stopping
        if f1 > best_f1 + 1e-4:
            best_f1 = f1
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break
    
    # Load best checkpoint for final eval
    best_ckpt, best_epoch = ckpt_manager.get_best()
    if best_ckpt is not None:
        model.load_state_dict(best_ckpt["model_state_dict"])
        model.to(device)
        print(f"\nLoaded best checkpoint from epoch {best_epoch}")
    
    # Final evaluation
    model.eval()
    all_pred, all_true, all_probs = [], [], []
    with torch.no_grad():
        for _, X, mask, Y in loader_test:
            X, mask = X.to(device), mask.to(device)
            logits = model(X, mask)
            all_probs.extend(F.softmax(logits, dim=1)[:, 1].cpu().numpy())
            all_pred.extend(logits.argmax(dim=1).cpu().numpy())
            all_true.extend(Y.numpy())
    
    # Print saved checkpoints
    print(f"\nSaved checkpoints (top 5):")
    for score, epoch in ckpt_manager.get_all_saved_epochs():
        print(f"  Epoch {epoch}: F1 = {score:.4f}")
    
    return {
        "model": model, "history": history, "best_epoch": best_epoch,
        "test_f1": f1_score(all_true, all_pred),
        "test_auroc": roc_auc_score(all_true, all_probs),
        "test_pred": np.array(all_pred),
        "test_true": np.array(all_true),
        "test_probs": np.array(all_probs),
        "ckpt_manager": ckpt_manager,
    }

## Train Model

In [6]:
results = train_tir_model(
    fasta_path=FASTA_PATH,
    label_path=LABEL_PATH,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
    device=DEVICE,
)

Device: mps

=== Loading data ===
Loaded 135751 TIR labels:
  TRUE (has TIR):  40424 (29.8%)
  FALSE (no TIR):  95327 (70.2%)
Matched 135751 sequences

Original: 95327 FALSE / 40424 TRUE
Target per class: 15000
Balanced: 15000 FALSE / 15000 TRUE = 30000 total

Train: 24000, Test: 6000
Parameters: 702,979
Batches/epoch: 750

=== Training ===


Ep 1/30:   0%|          | 0/750 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Results

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

epochs_range = range(1, len(results["history"]["train_loss"]) + 1)

axes[0].plot(epochs_range, results["history"]["train_loss"], 'b-', lw=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, results["history"]["val_f1"], 'g-', lw=2)
axes[1].axvline(results["best_epoch"], color="gray", ls="--", alpha=0.7)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1 Score")
axes[1].set_title("Validation F1")
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

axes[2].plot(epochs_range, results["history"]["val_auroc"], 'r-', lw=2)
axes[2].axvline(results["best_epoch"], color="gray", ls="--", alpha=0.7)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("AUROC")
axes[2].set_title("Validation AUROC")
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0, 1])

plt.tight_layout()
plt.savefig("tir_v2_training.png", dpi=150)
plt.show()

In [ ]:
# Final summary
print("=" * 50)
print("TIR v2 - FINAL RESULTS")
print("=" * 50)
print(f"Best Epoch: {results['best_epoch']}")
print(f"Test F1:    {results['test_f1']:.4f}")
print(f"Test AUROC: {results['test_auroc']:.4f}")
print()
print(classification_report(
    results["test_true"], results["test_pred"],
    target_names=["FALSE (no TIR)", "TRUE (has TIR)"]
))

# Confusion matrix
import seaborn as sns
cm = confusion_matrix(results["test_true"], results["test_pred"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["FALSE", "TRUE"], yticklabels=["FALSE", "TRUE"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("tir_v2_confusion.png", dpi=150)
plt.show()